# ViralCutter
Uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`

# Suporte em:
[![](https://dcbadge.limes.pink/api/server/tAdPHFAbud)](https://discord.gg/tAdPHFAbud)

# TODO📝
- [x] Release code
- [ ] Huggingface SpaceDemo
- [x] Two face in the cut
- [x] Custom caption and burn
- [x] Make the code faster
- [ ] More types of framing beyond 9:16

In [ ]:
#@title 🛠️ Install ViralCutter
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kodokbakar/ViralCutter.git"
BRANCH = "main"

PROJECT_DIR = Path("/content/ViralCutter")
VENV_DIR = PROJECT_DIR / ".venv"
PYTHON = VENV_DIR / "bin" / "python"


def run(title, command, cwd=None, env=None):
    """Run a command and stop immediately when it fails."""
    print(f"\n▶ {title}")
    subprocess.run(
        [str(value) for value in command],
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )


def uv_install(title, *packages):
    run(
        title,
        [
            "uv",
            "pip",
            "install",
            "--python",
            VENV_DIR,
            *packages,
        ],
        cwd=PROJECT_DIR,
    )


# ------------------------------------------------------------------
# 1. Clean and clone
# ------------------------------------------------------------------

print("🧹 Cleaning previous installation...")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

run(
    "Installing UV",
    ["python3", "-m", "pip", "install", "-q", "--upgrade", "uv"],
)

run(
    "Cloning ViralCutter",
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        BRANCH,
        REPO_URL,
        PROJECT_DIR,
    ],
)

run(
    "Checking repository revision",
    ["git", "log", "-1", "--oneline", "--decorate"],
    cwd=PROJECT_DIR,
)


# ------------------------------------------------------------------
# 2. System dependencies
# ------------------------------------------------------------------

apt_env = os.environ.copy()
apt_env["DEBIAN_FRONTEND"] = "noninteractive"

run(
    "Updating package list",
    ["apt-get", "update", "-qq"],
    env=apt_env,
)

run(
    "Installing system dependencies",
    [
        "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "xvfb",
        "libcudnn8",
        "pkg-config",
        "build-essential",
        "python3-dev",
        "libavformat-dev",
        "libavcodec-dev",
        "libavdevice-dev",
        "libavutil-dev",
        "libavfilter-dev",
        "libswscale-dev",
        "libswresample-dev",
    ],
    env=apt_env,
)


# ------------------------------------------------------------------
# 3. Virtual environment and PyTorch
# ------------------------------------------------------------------

run(
    "Creating virtual environment",
    ["uv", "venv", VENV_DIR, "--python", "python3"],
    cwd=PROJECT_DIR,
)

uv_install(
    "Installing NumPy compatibility stack",
    "numpy==1.26.4",
    "setuptools==69.5.1",
)

run(
    "Installing PyTorch CUDA 12.1",
    [
        "uv",
        "pip",
        "install",
        "--python",
        VENV_DIR,
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
        "torch==2.3.1+cu121",
        "torchvision==0.18.1+cu121",
        "torchaudio==2.3.1+cu121",
    ],
    cwd=PROJECT_DIR,
)


# ------------------------------------------------------------------
# 4. WhisperX compatibility stack
# ------------------------------------------------------------------

# PyAV 11 must be built against the FFmpeg development libraries.
uv_install(
    "Installing PyAV",
    "av==11.0.0",
)

# faster-whisper 1.0.0 requires tokenizers below 0.16.
uv_install(
    "Installing WhisperX",
    "whisperx==3.2.0",
    "faster-whisper==1.0.0",
    "ctranslate2==4.4.0",
    "transformers==4.35.2",
    "tokenizers==0.15.2",
    "huggingface-hub==0.24.7",
    "accelerate==0.26.1",
)


# ------------------------------------------------------------------
# 5. ViralCutter dependencies
# ------------------------------------------------------------------

run(
    "Installing ViralCutter dependencies",
    [
        "uv",
        "pip",
        "install",
        "--python",
        VENV_DIR,
        "-r",
        "requirements-colab.txt",
    ],
    cwd=PROJECT_DIR,
)

uv_install(
    "Installing computer vision dependencies",
    "insightface",
    "onnxruntime-gpu",
    "mediapipe>=0.10.0",
    "protobuf>=3.20,<5.0",
    "flatbuffers>=2.0",
)


# ------------------------------------------------------------------
# 6. Verify
# ------------------------------------------------------------------

verify_code = """
import sys
import av
import numpy
import torch
import tokenizers
import transformers
import ctranslate2
import faster_whisper
import whisperx
import gradio

print("Python:", sys.version.split()[0])
print("NumPy:", numpy.__version__)
print("PyAV:", av.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("CTranslate2:", ctranslate2.__version__)
print("Gradio:", gradio.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("WhisperX stack: OK")
"""

run(
    "Verifying installation",
    [PYTHON, "-u", "-c", verify_code],
    cwd=PROJECT_DIR,
)


# ------------------------------------------------------------------
# 7. Dependency check and virtual display
# ------------------------------------------------------------------

print("\n▶ Checking dependency consistency")

check = subprocess.run(
    [
        "uv",
        "pip",
        "check",
        "--python",
        VENV_DIR,
    ],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)

print(check.stdout or check.stderr)

if check.returncode == 0:
    print("✅ Dependencies are consistent.")
else:
    print("⚠️ Dependency warnings detected.")

if subprocess.run(
    ["pgrep", "-f", "Xvfb :1"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode != 0:
    subprocess.Popen(
        ["Xvfb", ":1", "-screen", "0", "2560x1440x24"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

os.environ["DISPLAY"] = ":1.0"
os.environ["MPLBACKEND"] = "Agg"

print("\n✅ ViralCutter installation completed.")
print(f"Repository: {PROJECT_DIR}")
print(f"Python: {PYTHON}")

✅ Instalação V7 Finalizada!
- Transformers 4.46.3 (Compatível com Alinhamento): INSTALADO
- Torch 2.3.1: ATIVO


In [ ]:
#@title 🩺 Runtime Doctor Preflight
import subprocess

%cd /content/ViralCutter

print("🩺 Running ViralCutter runtime doctor...")
subprocess.run(
    [
        "/content/ViralCutter/.venv/bin/python",
        "-c",
        "import sys; sys.path.insert(0, '/content/ViralCutter/webui'); import runtime_doctor; print(runtime_doctor.run_runtime_doctor())",
    ],
    check=False,
)

In [ ]:
#@title 🚀 Configuration and Run
import os
import subprocess
from google.colab import drive

%cd /content/ViralCutter

print("🔐 Mounting Google Drive...")
drive.mount("/content/drive")
drive_output_dir = "/content/drive/MyDrive/ViralCutter/VIRALS"
os.makedirs(drive_output_dir, exist_ok=True)

os.environ["VIRALCUTTER_OUTPUT_DIR"] = drive_output_dir

print(f"📁 ViralCutter outputs will be saved to: {drive_output_dir}")

os.system("Xvfb :1 -screen 0 2560x1440x8 &")
os.environ["DISPLAY"] = ":1.0"
os.environ["MPLBACKEND"] = "Agg"

print("🚀 Starting ViralCutter with the virtual environment...")
print("⚠️ Ignore harmless UserWarning messages. Wait for the public URL.")

!/content/ViralCutter/.venv/bin/python webui/app.py --colab

/content/ViralCutter
🚀 Iniciando ViralCutter usando o ambiente virtual...
⚠️ Ignore os avisos de 'UserWarning', aguarde o link public URL.
/content/ViralCutter/webui/app.py:204: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title=i18n("ViralCutter WebUI"), theme=gr.themes.Default(primary_hue="blue", neutral_hue="slate"), css=css) as demo:
Running in Colab mode. Generating public link...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7f3b04ec670707105b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#Créditos

Inspirado no [reels clips automator](https://github.com/eddieoz/reels-clips-automator) e no [YoutubeVideoToAIPoweredShorts](https://github.com/Fitsbit/YoutubeVideoToAIPoweredShorts)<br>

---
![Rafa.png](https://i.imgur.com/cGknQpU.png;base64)

Desenvolvido por **Rafa.Godoy**<br>
[ ![GitHub](https://img.shields.io/badge/github-%23121011.svg?style=for-the-badge&logo=github&logoColor=white) ](https://github.com/rafaelGodoyEbert)<br>
[ ![X](https://img.shields.io/twitter/url?url=https%3A%2F%2Ftwitter.com%2FGodoyEbert) ](https://twitter.com/GodoyEbert)<br>
[Instagram](https://www.instagram.com/rafael.godoy.ebert/)<br>
[ ![](https://dcbadge.vercel.app/api/server/aihubbrasil) ](https://discord.gg/aihubbrasil)

`0.1v Alpha`<br>

Apenas uma alternativa gratuita ao `opus.pro` e ao `vidyo.ai`<br>
